### import packages

In [23]:
#import required packages
import pandas as pd
import gensim
# NLTK Stop words
import pyLDAvis
from nltk.corpus import stopwords
import pickle
stop_words = stopwords.words('english')
from gensim import corpora
# Import Dataset
from gensim.utils import simple_preprocess
import pyLDAvis
import pyLDAvis.gensim
import spacy
import nltk
import numpy as np

In [2]:
# loads pipeline package for Language model (English)
nlp = spacy.load("en_core_web_sm")

In [130]:
#seting the maximum column width to display all columns
pd.set_option('display.max_colwidth',-1)

C:\Users\manasa\anaconda3\lib\site-packages\ipykernel_launcher.py:2: FutureWarning: Passing a negative integer is deprecated in version 1.0 and will not be supported in future version. Instead, use None to not limit the column width.
  


### read json file

In [131]:
#loads json file
jsn=json.loads(open('case_data.json').read())

In [132]:
 #converts json to dataframe
    data=pd.DataFrame.from_dict(pd.json_normalize(jsn,'data'))

In [133]:
#display first 4 rows
data.head(4)

,id,name,description,metrics,industries,created_at,updated_at,function
0,493,Retargeting,Retarget customers who have already expressed interest in your products or services,Cash flow\nReturn on marketing investment (ROMI),Broadcasting\nFilm Distribution\nMedia and Entertainment\nEnterprise Software\nFinTech\nBanking\nFinancial Services\nArt\nHome Decor\nFashion\nE-Commerce\nRecruiting\nDelivery\nFood and Beverage\nE-Commerce\nFood Processing\nLifestyle\nDietary Supplements\nE-Commerce\nAI Tools\nSupply Chain Management\nMarketplace\nAnalytics\nB2B\nImage Recognition\nDigital Media\nPhotography\nPredictive Analytics\nSEO\nAdvertising\nDirect Marketing\nEmail Marketing\nE-Commerce\nAnalytics\nEvents\nTicketing\nInternet\nTravel\nFashion\nShopping\nE-Commerce\nLifestyle\nFurniture\nFashion\nE-Commerce\nFashion\nE-Commerce\nFitness\nCooking\nHospitality\nMobile\nPersonalization\nFashion\nE-Commerce\nJewelry\nLifestyle\nCustomer Service\nRetail Technology\nInternet\nE-Commerce\nFashion\nManufacturing\nInternet\nE-Commerce\nRetail\nTelecommunications\nRetail\nMobile Devices\nCustomer Service\nLifestyle\nFashion\nRetail Technology\nSoftware Solutions\nE-Commerce\nRetail\nRetail\nOpen Source\nSoftware Solutions\nHardware\nDIY\n3D Printing\nReal Estate\nClassifieds\nE-Commerce\nNews,2020-02-03T15:08:21.159Z,2020-02-03T15:08:21.159Z,Marketing
1,494,Recommendation engine,"Also called recommendation personalization system or recommendation system, these systems leverage customer data to retain and upgrade customers with personalized recommendations via email, site search or other channels",Cash flow\nReturn on marketing investment (ROMI)\nEngagement & conversion\nB2C sales,Search Engine\nRetail\nShopping\nE-Commerce\nLifestyle\nFashion\nRetail\nFashion\nWomen's\nBeauty\nCosmetics\nRetail\nE-Commerce\nFashion,2020-02-03T15:08:21.212Z,2020-02-03T15:08:21.212Z,Marketing
2,495,Social analytics & automation,"Leverage Natural Language Processing and machine vision to analyze and act upon all content generated by your actual or potential customers on social media, surveys and reviews",Customer service effectiveness\nNet Promoter Score (NPS),Cloud Computing\nSoftware Solutions\nInformation Technology\nWeb Hosting\nIaaS\nManufacturing\nAutomotive\nTransportation\nVenture Capital\nFinance\nAngel Investment\nSocial Media\nMarketing\nAdvertising\nBusiness Information Systems\nBusiness Intelligence\nInternet\nAdvertising\nTest and Measurement\nVending and Concessions,2020-02-03T15:08:21.249Z,2020-02-03T15:08:21.249Z,Marketing
3,496,Product Information Management,Manage and improve all your product information centrally to improve product discoverability and appeal,Return on marketing investment (ROMI),Communications Infrastructure\nHardware\nEnterprise Software\nVenture Capital\nFinance\nAngel Investment\nE-Commerce Platforms\nFashion\nBeauty\nRetail\nE-Commerce\nRetail Technology,2020-02-03T15:08:21.270Z,2020-02-03T15:08:21.270Z,Marketing


In [256]:
#convert series to list
data_list=list(data.loc[:,'description'].values)

In [8]:
#check the number of features
data.shape

(122, 8)

### training 'description' feature to find the target

In [272]:
#copy of data
X=data.copy()
#process description feature to find the target
X=X['description']#,'name','industries','metrics']]

### stop words  (NLTK)

In [273]:
def strip_newline(series):
    return [review.replace('\n','') for review in series]

def sent_to_words(sentences):
    
    for sentence in sentences:
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
        
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]

def bigrams(words, bi_min=15, tri_min=10):
    bigram = gensim.models.Phrases(words, min_count = bi_min)
    bigram_mod = gensim.models.phrases.Phraser(bigram)
    return bigram_mod

In [274]:
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['come','order','try','go','get','make','drink','plate','dish','restaurant','place',
                  'would','really','like','great','service','came','got'])

def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]

def bigrams(words, bi_min=15, tri_min=10):
    bigram = gensim.models.Phrases(words, min_count = bi_min)
    bigram_mod = gensim.models.phrases.Phraser(bigram)
    return bigram_mod
    
def get_corpus(df):
    df['text'] = strip_newline(df)
    words = list(sent_to_words(df.text))
    words = remove_stopwords(words)
    bigram_mod = bigrams(words)
    bigram = [bigram_mod[review] for review in words]
    id2word = gensim.corpora.Dictionary(bigram)
    id2word.filter_extremes(no_below=10, no_above=0.35)
    id2word.compactify()
    corpus = [id2word.doc2bow(text) for text in bigram]
    return corpus, id2word, bigram

train_corpus, train_id2word, bigram_train = get_corpus(X)


### creating a models for corpus, bigrams, bag of words

In [197]:

with open('train_corpus.pkl', 'wb') as f:
    pickle.dump(train_corpus, f)
with open('train_id2word.pkl', 'wb') as f:
    pickle.dump(train_id2word, f)
with open('bigram_train.pkl', 'wb') as f:
    pickle.dump(bigram_train, f)


In [198]:
with open('train_corpus.pkl', 'rb') as f:
    train_corpus = pickle.load(f)
with open('train_id2word.pkl', 'rb') as f:
    train_id2word = pickle.load(f)
with open('bigram_train.pkl', 'rb') as f:
    bigram_train = pickle.load(f)

### LDA Model

In [199]:
import logging
import warnings
logging.basicConfig(filename='lda_model.log', format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    lda_train = gensim.models.ldamulticore.LdaMulticore(
                           corpus=train_corpus,
                           num_topics=20,
                           id2word=train_id2word,
                           chunksize=100,
                           workers=7, # Num. Processing Cores - 1
                           passes=50,
                           eval_every = 1,
                           per_word_topics=True)
    lda_train.save('lda_train.model')

### traning on LDA model

In [200]:
query = 'Predict risk of churn for individual customers/clients and recommend  renegotiation strategy '
vec_bow = train_id2word.doc2bow(query.lower().split(),return_missing=True)
print(vec_bow[1].keys())


dict_keys(['and', 'churn', 'customers/clients', 'for', 'individual', 'of', 'predict', 'recommend', 'renegotiation', 'risk', 'strategy'])


In [202]:
x = np.array(train_vecs)
y = np.array(data['function'])

kf = KFold(5, shuffle=True, random_state=42)
cv_lr_f1, cv_lrsgd_f1, cv_svcsgd_f1,  = [], [], []

for train_ind, val_ind in kf.split(x, y):
    # Assign CV IDX
    X_train, y_train = x[train_ind], y[train_ind]
    X_val, y_val = x[val_ind], y[val_ind]
    
    # Scale Data
    scaler = StandardScaler()
    X_train_scale = scaler.fit_transform(X_train)
    X_val_scale = scaler.transform(X_val)

    # Logisitic Regression
    lr = LogisticRegression(
        class_weight= 'balanced',
        solver='newton-cg',
        fit_intercept=True
    ).fit(X_train_scale, y_train)

    y_pred = lr.predict(X_val_scale)
    cv_lr_f1.append(f1_score(y_val, y_pred, average='micro'))
    
    # Logistic Regression SGD
    sgd = linear_model.SGDClassifier(
        max_iter=1000,
        tol=1e-3,
        loss='log',
        class_weight='balanced'
    ).fit(X_train_scale, y_train)
    
    y_pred = sgd.predict(X_val_scale)
    cv_lrsgd_f1.append(f1_score(y_val, y_pred, average='micro'))
    
    # SGD Modified Huber
    sgd_huber = linear_model.SGDClassifier(
        max_iter=1000,
        tol=1e-3,
        alpha=20,
        loss='modified_huber',
        class_weight='balanced'
    ).fit(X_train_scale, y_train)
    
    y_pred = sgd_huber.predict(X_val_scale)
    cv_svcsgd_f1.append(f1_score(y_val, y_pred, average='micro'))

print(f'Logistic Regression Val f1: {np.mean(cv_lr_f1):.3f} +- {np.std(cv_lr_f1):.3f}')
print(f'Logisitic Regression SGD Val f1: {np.mean(cv_lrsgd_f1):.3f} +- {np.std(cv_lrsgd_f1):.3f}')
print(f'SVM Huber Val f1: {np.mean(cv_svcsgd_f1):.3f} +- {np.std(cv_svcsgd_f1):.3f}')

Logistic Regression Val f1: 0.418 +- 0.076
Logisitic Regression SGD Val f1: 0.385 +- 0.082
SVM Huber Val f1: 0.099 +- 0.086


C:\Users\manasa\anaconda3\lib\site-packages\scipy\optimize\linesearch.py:327: LineSearchWarning: The line search algorithm did not converge
  warn('The line search algorithm did not converge', LineSearchWarning)
C:\Users\manasa\anaconda3\lib\site-packages\sklearn\utils\optimize.py:204: UserWarning: Line Search failed
  warnings.warn('Line Search failed')
C:\Users\manasa\anaconda3\lib\site-packages\scipy\optimize\linesearch.py:327: LineSearchWarning: The line search algorithm did not converge
  warn('The line search algorithm did not converge', LineSearchWarning)
C:\Users\manasa\anaconda3\lib\site-packages\sklearn\utils\optimize.py:204: UserWarning: Line Search failed
  warnings.warn('Line Search failed')
C:\Users\manasa\anaconda3\lib\site-packages\scipy\optimize\linesearch.py:327: LineSearchWarning: The line search algorithm did not converge
  warn('The line search algorithm did not converge', LineSearchWarning)
C:\Users\manasa\anaconda3\lib\site-packages\sklearn\utils\optimize.py:204:

### Coherence of  LDA model

In [234]:
from gensim.models import CoherenceModel
coherence_model_lda = CoherenceModel(model=lda_train, texts=bigram_train, dictionary=train_id2word, coherence='c_v')
coherence_ld = coherence_model_lda.get_coherence()
coherence_ld

0.2700564599662442

### displaying 10 of the 20 topics with 15 top words for each

In [112]:
lda_train.print_topics(20,num_words=15)[:10]

[(0,
  '0.622*"customers" + 0.334*"leverage" + 0.012*"machine" + 0.002*"advanced" + 0.002*"analytics" + 0.002*"sales" + 0.002*"ai" + 0.002*"learning" + 0.002*"use" + 0.002*"processing" + 0.002*"systems" + 0.002*"analyze" + 0.002*"data" + 0.002*"improve" + 0.002*"natural"'),
 (1,
  '0.059*"use" + 0.059*"machine" + 0.059*"improve" + 0.059*"learning" + 0.059*"processing" + 0.059*"analytics" + 0.059*"ai" + 0.059*"advanced" + 0.059*"customers" + 0.059*"analyze" + 0.059*"natural" + 0.059*"leverage" + 0.059*"data" + 0.059*"systems" + 0.059*"customer"'),
 (2,
  '0.314*"natural" + 0.314*"processing" + 0.239*"leverage" + 0.113*"machine" + 0.002*"customer" + 0.002*"customers" + 0.002*"satisfaction" + 0.002*"improve" + 0.002*"learning" + 0.002*"systems" + 0.002*"use" + 0.002*"analytics" + 0.002*"ai" + 0.002*"advanced" + 0.002*"sales"'),
 (3,
  '0.972*"sales" + 0.002*"analytics" + 0.002*"data" + 0.002*"leverage" + 0.002*"machine" + 0.002*"learning" + 0.002*"use" + 0.002*"processing" + 0.002*"ai" + 

In [113]:
topics_matrix = lda_train.show_topics(formatted=False, num_words=20)
topics_matrix = np.array(topics_matrix)

topic_words = topics_matrix[:,1]
for i in topic_words:
    print([str(word) for word in i])
    print()

["('data', 0.8749231)", "('analyze', 0.10713083)", "('customers', 0.001196561)", "('use', 0.0011964418)", "('improve', 0.0011964375)", "('ai', 0.0011964324)", "('advanced', 0.0011964233)", "('leverage', 0.00119642)", "('analytics', 0.0011964141)", "('sales', 0.0011963597)", "('learning', 0.0011963596)", "('processing', 0.0011963596)", "('machine', 0.0011963596)", "('systems', 0.0011963596)", "('customer', 0.0011963596)", "('natural', 0.0011963596)", "('satisfaction', 0.0011963596)"]

["('systems', 0.3506918)", "('processing', 0.35028413)", "('leverage', 0.17948696)", "('data', 0.008538667)", "('analyze', 0.008538667)", "('advanced', 0.008538667)", "('sales', 0.00853828)", "('ai', 0.00853828)", "('learning', 0.00853828)", "('use', 0.00853828)", "('customers', 0.00853828)", "('analytics', 0.00853828)", "('natural', 0.00853828)", "('machine', 0.00853828)", "('customer', 0.00853828)", "('improve', 0.00853828)", "('satisfaction', 0.00853828)"]

["('systems', 0.8183427)", "('learning', 0.155

C:\Users\manasa\anaconda3\lib\site-packages\ipykernel_launcher.py:2: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  


In [114]:
train_vecs = []
for i in range(len(X)-1):
    top_topics = lda_train.get_document_topics(train_corpus[i], minimum_probability=0.0)
    topic_vec = [top_topics[i][1] for i in range(20)]
    #topic_vec.extend([rev_train.iloc[i].real_counts]) # counts of reviews for restaurant
    #topic_vec.extend([len(X.iloc[i].text)]) # length review
    train_vecs.append(topic_vec)

### testing model with new unknown data

In [238]:
def get_bigram(df):
    """
    For the test data we only need the bigram data. This is a requirement due to 
    the shapes Gensim functions expect in the test-vector transformation below.
    With both these in hand, we can make the test corpus.
    """
    df['text'] = strip_newline(df)
    words = list(sent_to_words(df.text))
    words = remove_stopwords(words)
    bigram = bigrams(words)
    bigram = [bigram[review] for review in words]
#     lemma = lemmatization(bigram)
    return bigram

In [253]:
#input text to test
test_data=pd.Series('process operations on ai')

In [251]:
#get bigrams from test data
get_bigram(test_data)

[['process', 'operations', 'ai']]

In [243]:
#get lda model form traning 
lda_train = gensim.models.ldamulticore.LdaMulticore.load('lda_train.model')

In [244]:
# build bow from bigram text data
test_corpus = [train_id2word.doc2bow(text) for text in bigram_test]

In [247]:
#get text vectors
test_vecs = []
for i in range(len(test_data)-1):
    top_topics = lda_train.get_document_topics(test_corpus[i], minimum_probability=0.0)
    topic_vec = [top_topics[i][1] for i in range(20)]
    #topic_vec.extend([test_data.iloc[i].real_counts]) # counts of reviews for restaurant
    #topic_vec.extend([len(test_data.iloc[i].text)]) # length review
    test_vecs.append(topic_vec)

In [254]:
 #predict the target function on test/unknown data
sgd_huber.predict(test_vecs)

array(['Marketing'], dtype='<U17')